In [1]:
import os
import numpy as np
import pandas as pd

from astropy.time import Time
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroplan.observer import Observer

from rubin_scheduler.utils import m5_flat_sed
from rubin_scheduler.site_models import SeeingModel
from rubin_sim.skybrightness import SkyModel

# Limiting magnitude calculation for any RA/Dec/time/bandpass combination

Using the tools in rubin_scheduler and rubin_sim, and the formulas explained in [SMTN-002](https://smtn-002.lsst.io/#calculating-m5-values-in-the-lsst-operations-simulator) this notebook demonstrates how to calculate the expected limiting magnitude for any given visit. 

While the RA/Dec of a point on the sky is an important first piece of information, determining the airmass (and thus amount of atmospheric extinction, variation in delivered image quality, and variation in skybrightness) requires also specifying a time.  The m5 in all bandpasses is then shown in this notebook. 

Calculating these values for a large number of times would be computationally expensive primarily due to the skybrightness calculation; rubin_scheduler also holds pre-calculated skybrightness values in healpix grids for the time period covering the expected LSST survey. Use of the pre-calculated skybrightness is via rubin_scheduler.skybrightness_pre.SkyModelPre. For a small number of pointings, using SkyModel (as below) is more appropriate.


In [2]:
# Define observer location (for conversion to alt/az/pa + airmass)
observer = Observer.at_site("Rubin")
# Set up skybrightness model
skybrightness = SkyModel(mags=True)
# Set up the seeing model
seeing_model = SeeingModel()

In [3]:
# For the example, let's understand some convenient facts about a particular day then use those below
# Rubin DayObs
day_obs = Time("2026-01-05T12:00:00")
# What's the time at midnight? (or sun_set_time, etc.)
print(f"Sunset {observer.sun_set_time(day_obs, which='next').iso}, "\
      f"midnight {observer.midnight(day_obs, which='next').iso}, "\
      f"sunrise {observer.sun_rise_time(day_obs, which='next')}")

print(f"lunar illumination (1=full) {observer.moon_illumination(day_obs)} and phase (deg) (0=full) {np.degrees(observer.moon_phase(day_obs))}")

Sunset 2026-01-05 23:44:33.145, midnight 2026-01-06 04:48:46.130, sunrise 2461046.9117884086
lunar illumination (1=full) 0.9408005228504107 and phase (deg) (0=full) 28.16389618509472 deg


In [4]:
# Define RA/Dec/Time of observation

ra = 20
dec = -40
obs_coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)

# replace obs_time if desired
obs_time = Time("2026-01-05T02:30:00", scale='tai', format='isot')

In [5]:
# Check that RA/Dec is visible and calculate airmass
# Altitude < 15 degrees is inaccessible at Rubin

altaz = observer.altaz(obs_time, obs_coord)
airmass = altaz.secz.value

print(f"Altitude {altaz.alt.deg}, Azimuth {altaz.az.deg}, airmass {airmass}")

Altitude 47.73758816011089, Azimuth 242.5036341599272, airmass 1.351219046925091


In [6]:
# Find skybrightness at this location and time (assuming target is visible)
skybrightness.set_ra_dec_mjd(lon=ra, lat=dec, mjd=obs_time.mjd, degrees=True)

# Get skybackground magnitudes
sky_bg = skybrightness.return_mags()

sky_bg

{'u': array([20.35353374]),
 'g': array([19.61414496]),
 'r': array([19.56483526]),
 'i': array([19.43175002]),
 'z': array([18.93662049]),
 'y': array([18.24420234])}

In [7]:
# Determine the fwhm in the image (depends on airmass)
atmospheric_seeing = 0.6 #arcseconds
fwhm = seeing_model(atmospheric_seeing, airmass)['fwhmEff']
fwhm = dict([(band, fw) for band, fw in zip('ugrizy', fwhm)])
fwhm

{'u': np.float64(1.0815174585533427),
 'g': np.float64(1.0232807192560591),
 'r': np.float64(0.9700286618939677),
 'i': np.float64(0.9332902629867195),
 'z': np.float64(0.9089121292657261),
 'y': np.float64(0.8893946577933295)}

In [8]:
# Fill these values in to calculate the m5 appropriate for these conditions
# See also SMTN-002, "Calculating m5 values in the LSST Operations Simulator"
# https://smtn-002.lsst.io/#calculating-m5-values-in-the-lsst-operations-simulator

exp_time = {'u': 38, 'g': 30, 'r': 30, 'i': 30, 'z': 30, 'y': 30}

m5_visit = {}
for band in 'ugrizy':
    m5_visit[band] = m5_flat_sed(band, musky=sky_bg[band], fwhm_eff=fwhm[band], exp_time=exp_time[band], airmass=airmass, tau_cloud=0)

m5_visit

{'u': array([22.62045932]),
 'g': array([23.48229059]),
 'r': array([23.5194298]),
 'i': array([23.42779697]),
 'z': array([23.04024972]),
 'y': array([22.14674046])}